# VAR Dataset Construction (Annual Frequency)

Builds `var_dataset.csv` for System 1 VAR: `[rtb, xr, xb, y_nom, dp]`

**Frequency: Annual** — the VAR is estimated directly at annual frequency (one observation per calendar year). No quarterly-to-annual compounding needed.

**Raw data: quarterly** — variables are first constructed at quarterly frequency, then aggregated:
- **Returns** (`rtb`, `xr`, `xb`): sum of 4 quarterly log returns per calendar year
- **Levels** (`y_nom`, `dp`): end-of-year (Q4) value

**Target sample: ~1962 – 2025** (~64 annual observations).

The binding constraint is the GSW nominal yield curve (`feds200628.csv`): NSS parameters
start 1961-06-14. First full calendar year of returns is 1962.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("Thesisdata")

for fname in ["feds200628 (1).csv", "TB3MS.csv", "CPIAUCSL.csv", "ie_data.xls"]:
    status = "OK" if (DATA_DIR / fname).exists() else "MISSING"
    print(f"  [{status}] {fname}")

## 1. GSW Nominal Yield Curve → Bond Returns

Load `feds200628 (1).csv` (Gurkaynak, Sack & Wright fitted nominal yield curve).

**Key issue:** The `SVENY10` column only starts 1971-08, but the NSS shape parameters
(`BETA0`–`BETA3`, `TAU1`, `TAU2`) go back to 1961-06. We compute y(10) and y(9.75)
directly from the NSS formula for the full sample. Where both exist, the NSS-computed
y(10) matches `SVENY10` to within 0.00005 percentage points.

**NSS formula:**
$$y(n) = \beta_0 + \beta_1 \frac{1-e^{-n/\tau_1}}{n/\tau_1} + \beta_2\!\left(\frac{1-e^{-n/\tau_1}}{n/\tau_1} - e^{-n/\tau_1}\right) + \beta_3\!\left(\frac{1-e^{-n/\tau_2}}{n/\tau_2} - e^{-n/\tau_2}\right)$$

Before 1980, `TAU2` is missing and `BETA3 = 0` → reduces to Nelson-Siegel (3 params).

In [ ]:
# --- Load GSW daily data ---
gsw_cols = ["SVENY10", "BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]
gsw_raw = pd.read_csv(
    DATA_DIR / "feds200628 (1).csv",
    skiprows=9,
    index_col="Date",
    parse_dates=True,
    na_values="NA",
)[gsw_cols]
gsw_raw.replace(-999.99, np.nan, inplace=True)

print(f"GSW daily: {gsw_raw.index[0].date()} → {gsw_raw.index[-1].date()}  ({len(gsw_raw)} trading days)")
print(f"SVENY10 first valid:  {gsw_raw['SVENY10'].first_valid_index().date()}")
print(f"BETA0 first valid:    {gsw_raw['BETA0'].first_valid_index().date()}")
print(f"TAU2 first valid:     {gsw_raw['TAU2'].first_valid_index().date()}")
gsw_raw[["SVENY10", "BETA0", "BETA3", "TAU1", "TAU2"]].head(3)

In [ ]:
def nss_yield(n, row):
    """Nelson-Siegel-Svensson yield (% p.a.) at maturity n years.
    
    Falls back to 3-parameter Nelson-Siegel when TAU2 is NaN (BETA3=0 in early sample).
    """
    b0, b1, b2, b3 = row["BETA0"], row["BETA1"], row["BETA2"], row["BETA3"]
    t1, t2 = row["TAU1"], row["TAU2"]
    if any(pd.isna(v) for v in [b0, b1, b2, t1]):
        return np.nan
    x1 = n / t1
    term1 = (1 - np.exp(-x1)) / x1
    term2 = term1 - np.exp(-x1)
    result = b0 + b1 * term1 + b2 * term2
    # Add Svensson extension if TAU2 is available (post-1980)
    if not (pd.isna(t2) or pd.isna(b3)):
        x2 = n / t2
        term3 = (1 - np.exp(-x2)) / x2 - np.exp(-x2)
        result += b3 * term3
    return result

In [ ]:
# --- Resample to end-of-quarter and compute yields from NSS ---
gsw_q = gsw_raw.resample("QE-DEC").last()

gsw_q["y10_nss"]  = gsw_q.apply(lambda row: nss_yield(10,   row), axis=1)
gsw_q["y975_nss"] = gsw_q.apply(lambda row: nss_yield(9.75, row), axis=1)

# --- Validate: NSS-computed y(10) vs SVENY10 where both exist ---
both = gsw_q[["SVENY10", "y10_nss"]].dropna()
diff = (both["SVENY10"] - both["y10_nss"]).abs()

print(f"NSS y(10) available:  {gsw_q['y10_nss'].first_valid_index().date()} → "
      f"{gsw_q['y10_nss'].last_valid_index().date()}  "
      f"({gsw_q['y10_nss'].notna().sum()} quarters)")
print(f"SVENY10 available:    {gsw_q['SVENY10'].first_valid_index().date()} → "
      f"{gsw_q['SVENY10'].last_valid_index().date()}  "
      f"({gsw_q['SVENY10'].notna().sum()} quarters)")
print(f"\nValidation (overlap = {len(both)} quarters):")
print(f"  Max |SVENY10 - NSS y(10)| = {diff.max():.6f} pct pts")
print(f"  Mean                       = {diff.mean():.6f} pct pts")

In [ ]:
# --- Construct bond return and excess bond return ---
# Quarterly return on a 10-year zero-coupon bond:
#   Buy at end of quarter t-1 at y(10)_{t-1}, sell at end of quarter t
#   as a 9.75-year bond at y(9.75)_t.
#
#   r_bond_t = -9.75 * y(9.75)_t / 100  +  10 * y(10)_{t-1} / 100
#
# This is a log return (continuously compounded yields).

y10_lag = gsw_q["y10_nss"].shift(1)   # y(10) at end of previous quarter
y975    = gsw_q["y975_nss"]           # y(9.75) at end of current quarter

r_bond = -9.75 * y975 / 100 + 10 * y10_lag / 100

# Quick sanity check: bond return stats
r_bond_clean = r_bond.dropna()
print(f"r_bond available: {r_bond_clean.index[0].date()} → {r_bond_clean.index[-1].date()}  "
      f"({len(r_bond_clean)} quarters)")
print(f"  Mean:  {r_bond_clean.mean():.5f}  ({r_bond_clean.mean()*400:.2f}% p.a.)")
print(f"  Std:   {r_bond_clean.std():.5f}  ({r_bond_clean.std()*200:.2f}% p.a.)")  # *2 for annualizing std
print(f"  Min:   {r_bond_clean.min():.5f}")
print(f"  Max:   {r_bond_clean.max():.5f}")

## 2. T-Bill Rate + CPI → Real Bill Rate, Excess Bond Return, Nominal Yield

- **TB3MS**: 3-month T-bill rate (% p.a., FRED). Lagged one quarter → rate set at start of quarter.
- **CPIAUCSL**: CPI-U seasonally adjusted (FRED). Used for quarterly log inflation.

| Variable | Formula | Units |
|----------|---------|-------|
| `r_bill` | `TB3MS_{t-1} / 400` | Quarterly decimal |
| `pi_q` | `log(CPI_t / CPI_{t-1})` | Quarterly log inflation |
| `rtb` | `r_bill - pi_q` | Ex-post real bill rate |
| `xb` | `r_bond - r_bill` | Excess nominal bond return |
| `y_nom` | `y10_nss / 100` | 10-year nominal yield (annual decimal) |

In [ ]:
# --- Load T-bill rate and CPI ---
tbill = pd.read_csv(DATA_DIR / "TB3MS.csv", index_col="observation_date", parse_dates=True)
tbill.index = tbill.index.to_period("M").to_timestamp("M")  # month-end

cpi = pd.read_csv(DATA_DIR / "CPIAUCSL.csv", index_col="observation_date", parse_dates=True)
cpi.index = cpi.index.to_period("M").to_timestamp("M")

# Resample to end-of-quarter
tbill_q = tbill.resample("QE-DEC").last()
cpi_q   = cpi.resample("QE-DEC").last()

print(f"TB3MS:   {tbill_q.index[0].date()} → {tbill_q.last_valid_index().date()}  ({tbill_q['TB3MS'].notna().sum()} quarters)")
print(f"CPIAUCSL: {cpi_q.index[0].date()} → {cpi_q.last_valid_index().date()}  ({cpi_q['CPIAUCSL'].notna().sum()} quarters)")

In [ ]:
# --- Construct variables ---
# r_bill: nominal bill return earned during quarter t = rate SET at end of t-1
r_bill = tbill_q["TB3MS"].shift(1) / 400

# pi_q: quarterly log CPI inflation during quarter t
pi_q = np.log(cpi_q["CPIAUCSL"] / cpi_q["CPIAUCSL"].shift(1))

# rtb: ex-post real bill rate
rtb = r_bill - pi_q

# xb: excess nominal bond return (r_bond from Step 1 minus r_bill)
xb = r_bond - r_bill

# y_nom: 10-year nominal yield in annual decimal (SVENY10 / 100)
y_nom = gsw_q["y10_nss"] / 100

# --- Summary ---
for name, s in [("r_bill", r_bill), ("pi_q", pi_q), ("rtb", rtb), ("xb", xb), ("y_nom", y_nom)]:
    clean = s.dropna()
    print(f"{name:7s}: {clean.index[0].date()} → {clean.index[-1].date()}  "
          f"({len(clean)} Q)  mean={clean.mean():.6f}  std={clean.std():.6f}")

## 3. Shiller S&P 500 → Stock Returns and Dividend-Price Ratio

Source: `ie_data.xls` (Robert Shiller's online dataset).

| Column | Index | Content |
|--------|-------|---------|
| P | 1 | Nominal S&P 500 price |
| D | 2 | Trailing 12-month annual dividend |
| RTRP | 9 | Real Total Return Price (cumulative real total return index) |

| Variable | Formula | Units |
|----------|---------|-------|
| `xr` | `log(RTRP_t / RTRP_{t-1}) - rtb` | Quarterly log excess real stock return |
| `dp` | `log(D_t / P_t)` | Log dividend-price ratio (level) |

Note: `RTRP` is a real total return index (dividends reinvested, deflated by CPI),
so `log(RTRP_t / RTRP_{t-1})` is already a real return. Subtracting `rtb` (real bill rate)
gives the excess return.

In [ ]:
# --- Load Shiller data ---
shiller_raw = pd.read_excel(DATA_DIR / "ie_data.xls", sheet_name="Data", header=7, engine="xlrd")

def shiller_float_to_date(d):
    """Convert Shiller float date (1990.01 = Jan 1990) to Timestamp."""
    year = int(d)
    month = round((d - year) * 100)
    month = max(1, min(12, month if month > 0 else 1))
    return pd.Timestamp(year=year, month=month, day=1)

date_raw = pd.to_numeric(shiller_raw.iloc[:, 0], errors="coerce")
valid = date_raw.notna()
shiller = shiller_raw.loc[valid].copy()
shiller.index = date_raw[valid].apply(shiller_float_to_date)
shiller.index = shiller.index.to_period("M").to_timestamp("M")  # month-end
shiller.index.name = "Date"

# Positional indexing — robust against duplicate column names
P    = pd.to_numeric(shiller.iloc[:, 1], errors="coerce")  # nominal price
D    = pd.to_numeric(shiller.iloc[:, 2], errors="coerce")  # trailing 12-month dividend
RTRP = pd.to_numeric(shiller.iloc[:, 9], errors="coerce")  # real total return price

shiller_monthly = pd.DataFrame({"P": P, "D": D, "RTRP": RTRP}).dropna(how="any")
print(f"Shiller: {shiller_monthly.index[0].date()} → {shiller_monthly.index[-1].date()}  "
      f"({len(shiller_monthly)} months)")
shiller_monthly.tail(3)

In [ ]:
# --- Resample to end-of-quarter and construct xr, dp ---
P_q    = shiller_monthly["P"].resample("QE-DEC").last()
D_q    = shiller_monthly["D"].resample("QE-DEC").last()
RTRP_q = shiller_monthly["RTRP"].resample("QE-DEC").last()

# xr: excess real stock return = real stock return - real bill rate
#   log(RTRP_t / RTRP_{t-1}) is the real total return (RTRP is already deflated)
#   rtb is the real bill rate
xr = np.log(RTRP_q / RTRP_q.shift(1)) - rtb

# dp: log dividend-price ratio (D is trailing 12-month annual dividend)
dp = np.log(D_q / P_q)

# --- Summary ---
for name, s in [("xr", xr), ("dp", dp)]:
    clean = s.dropna()
    print(f"{name:7s}: {clean.index[0].date()} → {clean.index[-1].date()}  "
          f"({len(clean)} Q)  mean={clean.mean():.6f}  std={clean.std():.6f}")

## 4. Aggregate to Annual Frequency and Save `var_dataset.csv`

**Aggregation rules:**
- **Returns** (`rtb`, `xr`, `xb`): sum of 4 quarterly log returns within each calendar year. This gives the annual log return for each year (Q1+Q2+Q3+Q4).
- **Levels** (`y_nom`, `dp`): Q4 (end-of-year) value. These are state variables observed at the end of the period.

**Calendar year alignment:** Year `t` has:
- Returns earned during Q1–Q4 of year `t`
- Level variables observed at end of Q4 of year `t`

**Sample:** First full year with all 4 quarters available determines the start.
Quarterly `xb` starts 1961Q3, so first complete year is **1962**.

In [ ]:
# --- Assemble quarterly DataFrame first ---
var_q = pd.DataFrame({
    "rtb":   rtb,
    "xr":    xr,
    "xb":    xb,
    "y_nom": y_nom,
    "dp":    dp,
})
var_q.index.name = "date"
var_q = var_q.sort_index()

# Clean quarterly sample (all 5 non-NaN)
q_clean = var_q.dropna()
print(f"Quarterly (clean): {q_clean.index[0].date()} → {q_clean.index[-1].date()}  ({len(q_clean)} Q)")

# --- Aggregate to annual ---
return_cols = ["rtb", "xr", "xb"]
level_cols  = ["y_nom", "dp"]

# Group by calendar year
q_clean_year = q_clean.copy()
q_clean_year["year"] = q_clean_year.index.year

# Only keep years with exactly 4 quarters
qcount = q_clean_year.groupby("year").size()
full_years = qcount[qcount == 4].index
print(f"Full years (4 quarters each): {full_years[0]} – {full_years[-1]}  ({len(full_years)} years)")

q_full = q_clean_year[q_clean_year["year"].isin(full_years)]

# Returns: sum of 4 quarterly log returns
annual_returns = q_full.groupby("year")[return_cols].sum()

# Levels: Q4 value (last observation in year)
annual_levels = q_full.groupby("year")[level_cols].last()

# Combine
var_annual = pd.concat([annual_returns, annual_levels], axis=1)
var_annual.index.name = "year"

# Reorder columns to match System 1 ordering
var_annual = var_annual[["rtb", "xr", "xb", "y_nom", "dp"]]

print(f"\nAnnual dataset: {var_annual.index[0]} – {var_annual.index[-1]}  ({len(var_annual)} years)")
print(f"\nSummary statistics:")
display(var_annual.describe().round(6))

In [ ]:
# --- Save ---
out_path = Path("var_dataset.csv")
var_annual.to_csv(out_path)
print(f"Saved → {out_path}  ({out_path.stat().st_size / 1024:.1f} KB)")
print(f"Columns: {list(var_annual.columns)}")
print(f"Shape: {var_annual.shape}")
print(f"Frequency: ANNUAL (1 observation per calendar year)")
print()
print("First 5 rows:")
display(var_annual.head())
print()
print("Last 5 rows:")
display(var_annual.tail())

# --- Sanity checks on annual magnitudes ---
print("\n--- Annual magnitude sanity checks ---")
for col in return_cols:
    m = var_annual[col].mean()
    s = var_annual[col].std()
    print(f"  {col:5s}: mean={m:+.4f} ({m*100:+.2f}%)  std={s:.4f} ({s*100:.2f}%)")
print(f"  y_nom: mean={var_annual['y_nom'].mean():.4f}  ({var_annual['y_nom'].mean()*100:.2f}% p.a.)")
print(f"  dp:    mean={var_annual['dp'].mean():.4f}")

## 5. Validation

1. **Quarterly-to-annual consistency** — verify that annual returns = sum of 4 quarterly returns
2. **Annual VAR estimation** — estimate restricted VAR(1) directly on annual data, check parameter magnitudes
3. **Comparison with compounded quarterly VAR** — ensure direct annual estimation gives similar results to the old quarterly→annualize pipeline

In [ ]:
# === 1. Quarterly-to-annual consistency ===
print("=" * 60)
print("1. QUARTERLY-TO-ANNUAL CONSISTENCY CHECK")
print("=" * 60)

# Recompute annual returns from quarterly to verify
for col in return_cols:
    annual_check = q_full.groupby("year")[col].sum()
    diff = (var_annual[col] - annual_check).abs()
    print(f"  {col:5s}: max|annual - sum(quarterly)| = {diff.max():.2e}  (should be ~0)")

# Verify level columns are Q4 values
for col in level_cols:
    q4_vals = q_full[q_full.index.quarter == 4].set_index("year")[col]
    diff = (var_annual[col] - q4_vals).abs()
    print(f"  {col:5s}: max|annual - Q4 value| = {diff.max():.2e}  (should be ~0)")

print("\n--- Annual return magnitudes (sanity vs known benchmarks) ---")
print(f"  rtb mean: {var_annual['rtb'].mean()*100:+.2f}%  (expect ~0.5-1.5% real)")
print(f"  xr  mean: {var_annual['xr'].mean()*100:+.2f}%  (expect ~5-7% equity premium)")
print(f"  xb  mean: {var_annual['xb'].mean()*100:+.2f}%  (expect ~1-3% term premium)")
print(f"  xr  std:  {var_annual['xr'].std()*100:.2f}%  (expect ~15-20%)")
print(f"  xb  std:  {var_annual['xb'].std()*100:.2f}%  (expect ~8-12%)")

In [ ]:
# === 2. Annual VAR estimation (restricted) ===
print("=" * 60)
print("2. ANNUAL VAR(1) ESTIMATION — RESTRICTED")
print("=" * 60)

sys1_cols = ["rtb", "xr", "xb", "y_nom", "dp"]
state_indices = [0, 3, 4]  # rtb, y_nom, dp
state_cols = [sys1_cols[i] for i in state_indices]

data = var_annual[sys1_cols].copy()
n = len(sys1_cols)
Y = data.iloc[1:].to_numpy()
X_state = data[state_cols].iloc[:-1].to_numpy()
T_eff = Y.shape[0]
X = np.column_stack([np.ones(T_eff), X_state])
coeffs, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
Y_hat = X @ coeffs
resid = Y - Y_hat
dof = T_eff - X.shape[1]
Omega = (resid.T @ resid) / dof
const = coeffs[0, :]
slope_mat = coeffs[1:, :]
Phi = np.zeros((n, n))
for k, j in enumerate(state_indices):
    Phi[:, j] = slope_mat[k, :]
z_bar = np.linalg.solve(np.eye(n) - Phi, const)

print(f"Sample: {data.index[0]} – {data.index[-1]}  (T={T_eff} annual obs)")
print(f"\nPhi (transition matrix, annual):")
print(pd.DataFrame(Phi, index=sys1_cols, columns=sys1_cols).round(4).to_string())

print(f"\nPhi diagonal (state persistence):")
for i, col in enumerate(sys1_cols):
    print(f"  {col:5s}: {Phi[i,i]:+.4f}")

print(f"\nOmega diagonal (annual innovation variance):")
for i, col in enumerate(sys1_cols):
    print(f"  {col:5s}: {Omega[i,i]:.6f}  (std={np.sqrt(Omega[i,i]):.4f})")

print(f"\nz_bar (unconditional means):")
for i, col in enumerate(sys1_cols):
    print(f"  {col:5s}: {z_bar[i]:+.6f}")

print(f"\nResidual correlations:")
D_inv = np.diag(1.0 / np.sqrt(np.diag(Omega)))
corr = D_inv @ Omega @ D_inv
print(pd.DataFrame(corr, index=sys1_cols, columns=sys1_cols).round(4).to_string())

# Key economic checks
print(f"\n--- Key parameter checks ---")
# M = Sigma_rs @ inv(Sigma_ss) conditioning matrix
s_idx = np.array(state_indices)
r_idx = np.array([1, 2])
Omega_rs = Omega[np.ix_(r_idx, s_idx)]
Omega_ss = Omega[np.ix_(s_idx, s_idx)]
M = Omega_rs @ np.linalg.inv(Omega_ss)
print(f"M (conditioning matrix, annual):")
state_names = [sys1_cols[i] for i in state_indices]
ret_names = [sys1_cols[i] for i in r_idx]
print(pd.DataFrame(M, index=ret_names, columns=state_names).round(4).to_string())
print(f"  M[xb, y_nom] = {M[1, 1]:.2f}  (expect ≈ -9.75, i.e. ~bond duration)")

In [ ]:
# === 3. Comparison: direct annual vs compounded quarterly VAR ===
# Estimate quarterly VAR, then compound to annual, and compare with direct annual estimation.
print("=" * 60)
print("3. DIRECT ANNUAL vs COMPOUNDED QUARTERLY VAR")
print("=" * 60)

# --- Quarterly VAR estimation (restricted) ---
q_data = q_clean[sys1_cols].copy()
n = len(sys1_cols)
Y_q = q_data.iloc[1:].to_numpy()
X_q_state = q_data[state_cols].iloc[:-1].to_numpy()
T_q = Y_q.shape[0]
X_q = np.column_stack([np.ones(T_q), X_q_state])
coeffs_q, _, _, _ = np.linalg.lstsq(X_q, Y_q, rcond=None)
Y_hat_q = X_q @ coeffs_q
resid_q = Y_q - Y_hat_q
dof_q = T_q - X_q.shape[1]
Omega_q = (resid_q.T @ resid_q) / dof_q
const_q = coeffs_q[0, :]
slope_q = coeffs_q[1:, :]
Phi_q = np.zeros((n, n))
for k, j in enumerate(state_indices):
    Phi_q[:, j] = slope_q[k, :]
z_bar_q = np.linalg.solve(np.eye(n) - Phi_q, const_q)

# --- Compound quarterly → annual (h=4) ---
s_idx = list(np.array(state_indices))
r_idx = [1, 2]
ns = len(s_idx)

Phi_11_q = Phi_q[np.ix_(s_idx, s_idx)]
Phi_21_q = Phi_q[np.ix_(r_idx, s_idx)]
Omega_ss_q = Omega_q[np.ix_(s_idx, s_idx)]
Omega_rs_q = Omega_q[np.ix_(r_idx, s_idx)]
Omega_rr_q = Omega_q[np.ix_(r_idx, r_idx)]
Omega_sr_q = Omega_rs_q.T
c_s_q = const_q[s_idx]
c_r_q = const_q[r_idx]

h = 4
In = np.eye(ns)
P = [In.copy()]
for _ in range(h - 1):
    P.append(P[-1] @ Phi_11_q)
C = [In.copy()]
for k in range(1, h):
    C.append(C[k-1] + P[k])

Phi_11_comp = P[h-1] @ Phi_11_q
Phi_21_comp = Phi_21_q @ C[h-1]
Omega_ss_comp = sum(P[k] @ Omega_ss_q @ P[k].T for k in range(h))

# --- Compare ---
print(f"\nQuarterly VAR: T={T_q} obs,  Annual VAR: T={T_eff} obs")

print(f"\nPhi_11 diagonal (state persistence):")
print(f"  {'':5s}  {'Direct':>10s}  {'Compounded':>10s}  {'Diff':>10s}")
for i, col in enumerate(state_cols):
    d = Phi[state_indices[i], state_indices[i]]
    c = Phi_11_comp[i, i]
    print(f"  {col:5s}: {d:+10.4f}  {c:+10.4f}  {d-c:+10.4f}")

print(f"\nPhi_21 (return loadings on states):")
print(f"  {'':8s}  {'Direct':>10s}  {'Compounded':>10s}  {'Diff':>10s}")
for i, rn in enumerate(["xr", "xb"]):
    for j, sn in enumerate(state_cols):
        d = Phi[r_idx[i], state_indices[j]]
        c = Phi_21_comp[i, j]
        print(f"  {rn}←{sn}: {d:+10.4f}  {c:+10.4f}  {d-c:+10.4f}")

print(f"\nOmega_ss diagonal (state innovation variance):")
print(f"  {'':5s}  {'Direct':>12s}  {'Compounded':>12s}  {'Ratio':>8s}")
for i, col in enumerate(state_cols):
    d = Omega[state_indices[i], state_indices[i]]
    c = Omega_ss_comp[i, i]
    print(f"  {col:5s}: {d:12.6f}  {c:12.6f}  {d/c:8.4f}")

print(f"\nz_bar comparison:")
print(f"  {'':5s}  {'Direct':>10s}  {'Quarterly':>10s}  {'Diff':>10s}")
for i, col in enumerate(sys1_cols):
    print(f"  {col:5s}: {z_bar[i]:+10.6f}  {z_bar_q[i]:+10.6f}  {z_bar[i]-z_bar_q[i]:+10.6f}")